# 08 - Reproduction check against the original run

The analysis was first run with an earlier, Portuguese-language version of this code (provenance in
provenance/original_run/; its analysis plan was frozen on 2026-11-20 before the Greek labels were read). This
repository is a refactored English translation. This notebook compares every key number of the current run with the
value reported by the original run.

- Most original values were reported with 2–3 decimals: a value matches when it rounds to the reported value.
  Values reported as "< 0.001" match when the new value is below 0.001. Some values (AUCs of Table 2b, the null
  bootstrap, the design, the exploratory correlations) were stored at full precision and must agree to 1e-9.
- With the same data and the same library versions (see requirements.txt), all compared values should match.
  Different library versions or operating systems can change the last decimals of the Random Forest results.
- A quick test run (quick_test_mode) uses fewer trees and replicates, so many values will differ; that is
  expected.
- Table S7 (included vs excluded) was intentionally changed after the original run (it now compares participants of
  the same diagnostic group) and is not part of the check.

**Output:** outputs/reproduction_check.csv.

In [ ]:
# Setup: make the eegad package importable (also when it was not installed with `pip install -e .`)
import sys
import warnings
from pathlib import Path

warnings.filterwarnings("ignore", category=RuntimeWarning)   # MNE notices about head size and 'boundary' events
warnings.filterwarnings("ignore", category=FutureWarning)

_root = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "config" / "analysis_plan.json").exists()), None)
if _root is None:
    raise FileNotFoundError("Open this notebook from inside the repository (eeg-ad-demographics/notebooks).")
if str(_root / "src") not in sys.path:
    sys.path.insert(0, str(_root / "src"))

In [ ]:
import pandas as pd
from IPython.display import display

from eegad.config import load_context
from eegad.reproduction import compare, load_new, load_original, summary
from eegad.submission import build_share_zip

ctx = load_context("08")
original = load_original(ctx.repo / "provenance" / "original_run" / "key_results.json")
new = load_new(ctx.out / "results" / "key_results.json")
tab = compare(new, original)
tab.to_csv(ctx.out / "reproduction_check.csv", index=False)
build_share_zip(ctx.out)                     # share.zip now also carries reproduction_check.csv
print(summary(tab))
if ctx.quick:
    print("\nQuick test mode: differences are expected.")
pd.set_option("display.max_rows", 50)
print("\nValues that differ:")
display(tab[tab.status == "differs"])
print("\nOriginal values missing in this run:")
display(tab[tab.status == "missing in this run"])